# Donnan-Manning Sorption Model

Predicts salt sorption into a charged membrane using Donnan equilibrium plus
non-ideal activity coefficients: the external solution via the **Pitzer model**, and
the membrane phase via **Manning's counter-ion condensation theory**.

Two modes, both computed when you give enough input:
- **Predictive** — give a value for `b` (Å) and get the predicted sorption curve.
- **Fitted** — give measured Csm,w data and the notebook fits `b` to your data
  (minimizing RMS log error), then reports the fit and the derived condensation
  parameter `xi'`.

Python port of `Donnan_Manning.m`, `Manning.m`, `Manning_b_Fitter.m`, and `Pitzer.m`
from the original MATLAB code. See `sorption_models.py` in this folder for the math.

**Note:** the original MATLAB `Pitzer.m` had a typo (`alhpa2` instead of `alpha2`) that
left a variable undefined for non-monovalent salts (anything where neither ion is
+/-1). This port fixes that so non-monovalent salts work; see the docstring at the top
of `sorption_models.py` for details.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

from sorption_models import run_donnan_manning, load_pitzer_params, list_salts
from gui_helpers import EditableTable, labeled

params_df = load_pitzer_params("Pitzer Params.xlsx")
salts = list_salts(params_df)


## 1. Membrane and salt parameters

- **zg, zc, zA**: valences of the fixed charge group, counter-ion, and co-ion.
- **Phiw (DI)**, **CAm,w (DI)**: membrane water fraction and fixed-charge concentration
  equilibrated with DI water.
- **Salt**: must match a name in `Pitzer Params.xlsx`, or choose `MM` to use the
  Modified Manning model (Galizia et al.) instead of the full Pitzer solution model.
- **T (ºC)**: temperature.
- **b (Å)**: Manning parameter for the *predictive* model. Leave at 0 (or clear it) if
  you only want the fitted model.

In [ ]:
membrane_name = widgets.Text(value="", placeholder="e.g. C2VI2 Photo")
zg = widgets.FloatText(value=-1)
zc = widgets.FloatText(value=1)
zA = widgets.FloatText(value=1)
phiw_DI = widgets.FloatText(value=0.5)
CAmw_DI = widgets.FloatText(value=5.0)
salt = widgets.Dropdown(options=salts, value="NaCl" if "NaCl" in salts else salts[0])
T = widgets.FloatText(value=25)
b_text = widgets.Text(value="", placeholder="leave blank to skip the predictive model")

scalar_form = widgets.VBox([
    labeled(membrane_name, "Membrane name (optional)"),
    labeled(zg, "zg (fixed charge valence)"),
    labeled(zc, "zc (counter-ion valence)"),
    labeled(zA, "zA (co-ion valence)"),
    labeled(phiw_DI, "Phiw (DI) (-)"),
    labeled(CAmw_DI, "CAm,w (DI) (m)"),
    labeled(salt, "Salt"),
    labeled(T, "T (ºC)"),
    labeled(b_text, "b (Å) [optional, for predictive model]"),
])
display(scalar_form)


## 2. Concentration-dependent data

One row per external salt concentration.

- **Css (m)** — required.
- **phiw_s (-)** — optional; leave the whole column blank to reuse Phiw (DI) for every point.
- **Csmw measured (m)** — optional; provide it (for every row) to enable the *fitted*
  model and RMSLE reporting for the predictive model.

In [3]:
table = EditableTable(["Css (m)", "phiw_s (-) [optional]", "Csmw measured (m) [optional]"], n_rows=3)
display(table.widget)



## 3. Compute

In [ ]:
compute_btn = widgets.Button(description="Compute", button_style="primary")
out = widgets.Output()

def on_compute(_btn):
    with out:
        clear_output(wait=True)
        try:
            Css = table.get_column(0)
            if not Css:
                print("Enter at least one row with a Css value.")
                return
            phiw_s = table.get_column(1)
            Csmw_meas = table.get_column(2)

            b_str = b_text.value.strip()
            b_val = float(b_str) if b_str else None

            if b_val is None and not Csmw_meas:
                print("Provide b (predictive) and/or measured Csmw data (fitted) to compute anything.")
                return

            title = membrane_name.value or "Donnan-Manning model"
            result = run_donnan_manning(
                salt.value, zg.value, zc.value, zA.value, phiw_DI.value, CAmw_DI.value,
                T.value, Css, params_df, phiw_s or None, Csmw_meas or None, b=b_val,
            )

            print(f"=== {title} ({salt.value}, {T.value} \u00baC) ===")

            fig, ax = plt.subplots(figsize=(5, 4))
            shown_measured = False

            if "predictive" in result:
                pred = result["predictive"]
                print(f"\n-- Predictive (b = {pred['b']:.4g} \u00c5) --")
                display(pred["table"])
                if pred["rmsle"] is not None:
                    print(f"RMSLE (vs. measured data): {pred['rmsle']:.4g}")
                ax.plot(pred["table"]["Css (m)"], pred["table"]["Csm,w DM Predicted (m)"], "o-", label="DM Predicted")
                if "Csm,w measured (m)" in pred["table"].columns:
                    ax.plot(pred["table"]["Css (m)"], pred["table"]["Csm,w measured (m)"], "s", label="Measured")
                    shown_measured = True

            if "fitted" in result:
                fit = result["fitted"]
                print(f"\n-- Fitted --\nFitted b = {fit['b_fit']:.4g} \u00c5    Fitted xi' = {fit['xip_fit']:.4g}")
                display(fit["table"])
                print(f"RMSLE (fit quality): {fit['rmsle']:.4g}")
                ax.plot(fit["table"]["Css (m)"], fit["table"]["Csm,w DM Fitted (m)"], "^-", label="DM Fitted")
                if not shown_measured:
                    ax.plot(fit["table"]["Css (m)"], fit["table"]["Csm,w measured (m)"], "s", label="Measured")

            ax.set_xlabel("Css (m)")
            ax.set_ylabel("Csm,w (m)")
            ax.set_title(title)
            ax.legend()
            ax.set_xscale('log')
            ax.set_yscale('log')
            fig.tight_layout()
            plt.show()
        except ValueError as e:
            print(f"Input error: {e}")

compute_btn.on_click(on_compute)
display(widgets.VBox([compute_btn, out]))

